# Momentum Strategy Code vvv

#### Import all modules, packages, tools to be used in the code.

In [ ]:
import numpy as np
import pandas as pd
import pandas_datareader as pdr
import matplotlib.pyplot as plt
import mplfinance as mpf
import seaborn as sns
import itertools
import os
import re
import datetime 
from zipfile import ZipFile
import pandas_gbq
import janitor
import shutil
import glob
from google.cloud import bigquery
from google.oauth2 import service_account

In [ ]:
# pd.set_option('display.max_columns', None)

In [ ]:
pd.set_option('display.max_rows', 30)
pd.set_option('display.min_rows', 30)

* Read in ES data to ES and change the column names to make sense. 
* Drop NA's to avoid issues, convert date data type, 
* sort date appropriately, add day number, set index to date.

In [ ]:
ES = pd.read_csv('ES_Granular/ES_continuous_adjusted_1hour.txt', names = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume'])
ES.dropna(inplace=True)
# ES

In [ ]:
# ES = pd.read_csv('HistoricalData_ES.csv') #reads in ES data to ES
# ES['Close'] = ES['Close/Last'] #copies said column
# ES.drop(columns = ['Close/Last'], inplace = True) #drops 'columns' from dataframe, inplace makes it permanent
# ES = ES[['Date', 'Open', 'Close', 'High', 'Low', 'Volume']] #orders columns
# ES.dropna(inplace=True) #drops records that we can't really use 
# ES #prints top and bottom five records of dataframe

In [ ]:
ES['Date']=pd.to_datetime(ES.Date) #converts Date columns to datetime format
ES = ES.sort_values(['Date'], ascending = True) #sorts by Date column so earliest is first
# ES

In [ ]:
day = np.arange(1, len(ES) + 1) #creates day column
ES['Day'] = day #adds day column to dataframe
#ES.reset_index(drop = True, inplace = True)
ES = ES.set_index('Date')

#ES

In [ ]:
# #adjust the volume ma here
# vma = 5
ES['candle_average']=(ES['High']+ ES['Low']+ES['Open']+ES['Close'])/4 #creates average of our price metrics
# ES['vma']=ES['Volume'].rolling(vma).mean()#.shift() #creates volume moving average using vma as input
# #ES

In [ ]:
# Creating a copy of the table for weighted ma calculations
ESw = ES.copy()
ESe = ES.copy()

# Simple Moving Average Iteration

In [ ]:
LongMAs = list(range(1, 51))
SignalMAs = list(range(1, 51))
prix_met = 'Open'
prix_met_2 = 'Close'

### These cells iterate through lists of moving average periods and creates a column for each.

In [ ]:
for ma in LongMAs :
    ES[ma] = ES[prix_met].rolling(ma).mean()
#     break

In [ ]:
# ES

In [ ]:
new_names_lma = []
old_names_lma = []

for ma in LongMAs :
    new_names_lma.append(str(ma)+'_lma')
    old_names_lma.append(ma)
print(new_names_lma)
print(old_names_lma)

ES.rename(columns=dict(zip(old_names_lma, new_names_lma)), inplace = True)

In [ ]:
# ES

In [ ]:
for ma in SignalMAs :
    ES[ma] = ES[prix_met].rolling(ma).mean()

new_names_sma = []
old_names_sma = []

for ma in SignalMAs :
    new_names_sma.append(str(ma)+'_sma')
    old_names_sma.append(ma)
print(new_names_sma)
print(old_names_sma)

ES.rename(columns=dict(zip(old_names_sma, new_names_sma)), inplace = True)

In [ ]:
# Defragging ES
ES = ES.copy()
ES

### Next we'll iterate through our lma and sma columns and generate pairwise signals.

In [ ]:
list(itertools.product(new_names_lma, new_names_sma))

In [ ]:
# This is the hog.
signal_cols = []

for lname in new_names_lma :
    for sname in new_names_sma :
#         if pd.to_numeric(re.findall(r'\d+', lname)) > pd.to_numeric(re.findall(r'\d+', sname)) :
        ES[f"{lname} {sname} signal"] = np.where(ES[sname] > ES[lname], 1, 0) #long signal
        ES[f"{lname} {sname} signal"] = np.where(ES[sname] < ES[lname], -1, ES[f"{lname} {sname} signal"]) #short signal
        ES.dropna(inplace=True) #drops records that we can't really use due to nature of sma's
        signal_cols.append(f"{lname} {sname} signal")

In [ ]:
signal_cols

In [ ]:
ES = ES.copy()
ES

# Subsetting High Low Slope Code

In [ ]:
slope_period = 2
ES['Day_shift'] = ES['Day'].shift(slope_period)
ES['ca_shift'] = ES['candle_average'].shift(slope_period)
ES['ca_slope'] = (ES['candle_average'] - ES['candle_average'].shift(slope_period))/(ES['Day'] - ES['Day'].shift(slope_period))
ES

In [ ]:

ES['Day_shift'] = ES['Day'].shift(slope_period)
ES['open_shift'] = ES['Open'].shift(slope_period)
ES['open_slope'] = (ES['Open'] - ES['Open'].shift(slope_period))/(ES['Day'] - ES['Day'].shift(slope_period))
ES

In [ ]:
# This code obviously needs some work

periods = 30

ES['rolling_high'] = ES['candle_average'].rolling(window=periods).max()
ES['rolling_low'] = ES['candle_average'].rolling(window=periods).min()

# Identify the peaks and troughs in the price movement
ES['peak'] = np.where(ES['candle_average'] == ES['rolling_high'], 1, 0)
ES['trough'] = np.where(ES['candle_average'] == ES['rolling_low'], -1, 0)

In [ ]:
# Calculate the number of iterations required to split the DataFrame into chunks of 100 rows

table_size = 10000

num_iterations = len(ES) // table_size + 1
subset_filenames = []

# Initialize an empty dictionary to store the subsets
subset_dict = {}

# Loop over the DataFrame and create a new DataFrame with 100 rows in each iteration
for i in range(num_iterations):
    start_idx = i * table_size  # Calculate the starting index for the subset
    end_idx = (i + 1) * table_size  # Calculate the ending index for the subset
    subset_ES = ES.iloc[start_idx:end_idx]  # Subset the DataFrame using iloc
    
    # Add the subset DataFrame to the dictionary with a key that includes the iteration index
    subset_dict[f'subset_{i}'] = subset_ES
    
    filename = f'subset_{i}'
    subset_filenames.append(filename)

In [ ]:
subset = 'subset_950'
subset_dict[subset]

In [ ]:
# print(subset_filenames)

In [ ]:
# This code obviously needs some work


columns = ['Open', 'High', 'Low', 'Close', 'Volume']

# Create the candlestick chart using the mpf.plot function
mpf.plot(
    subset_dict[subset][columns], 
    mav=(2,10), 
    volume = True, 
    datetime_format='%Y-%m-%d %H:%M:%S', 
    type='candle', 
    title='Candlestick Chart',
    figsize = (20,10),
    addplot=[
        mpf.make_addplot(subset_dict[subset]['peak'] == 1, type='scatter', color='green', markersize=200, marker='^'),
        mpf.make_addplot(subset_dict[subset]['trough'] == -1, type='scatter', color='red', markersize=200, marker='v')
    ]
)


In [ ]:
ES

# Weighted Moving Average Iteration

### Defining functions for both the fast WMA and slow WMA. This was only necessary due to the naming of the columns.

In [ ]:
data = ESw['Open']

def weighted_moving_average(data, period):
    
    weights = []
    weights_calc = []

    for i in range(period + 1) :
        pd.to_numeric(weights_calc.append(i))
    for i in weights_calc :
        weights.append(i/sum(weights_calc))   
    weights.pop(0)
    
    wma = []
    for i in range(len(data) - period + 1):
        wma.append(sum(data[i:i+period] * weights) / sum(weights))
#     return wma
    wma = [float('nan')] * (period - 1) + wma
    ESw[f"{period}_lwma"] = wma
    
def weighted_moving_average_s(data, period):
    
    weights = []
    weights_calc = []

    for i in range(period + 1) :
        pd.to_numeric(weights_calc.append(i))
    for i in weights_calc :
        weights.append(i/sum(weights_calc))   
    weights.pop(0)
    
    wma = []
    for i in range(len(data) - period + 1):
        wma.append(sum(data[i:i+period] * weights) / sum(weights))
#     return wma
    wma = [float('nan')] * (period - 1) + wma
    ESw[f"{period}_swma"] = wma

#### Applying the functions to the weighted data set and creating a defrag copy.

In [ ]:
# Just for reference
# LongMAs = list(range(1, 51)) # change this back to 51
# SignalMAs = list(range(1, 51))
# prix_met = 'Open'
# prix_met_2 = 'Close'

for ma in LongMAs :
    weighted_moving_average(data, ma)

for ma in SignalMAs :
    weighted_moving_average_s(data, ma)    

In [ ]:
ESw = ESw.copy()
ESw

In [ ]:
l_wma_names = []
for period in LongMAs :
    l_wma_names.append(f"{period}_lwma")

s_wma_names = []
for period in SignalMAs :
    s_wma_names.append(f"{period}_swma")

In [ ]:
l_wma_names

In [ ]:
s_wma_names

In [ ]:
# This is the hog. It still needs to be changed to weighted
signal_cols_w = []

for lname in l_wma_names :
    for sname in s_wma_names :
#         if pd.to_numeric(re.findall(r'\d+', lname)) > pd.to_numeric(re.findall(r'\d+', sname)) :
        ESw[f"{lname} {sname} signal"] = np.where(ESw[sname] > ESw[lname], 1, 0) #long signal
        ESw[f"{lname} {sname} signal"] = np.where(ESw[sname] < ESw[lname], -1, ESw[f"{lname} {sname} signal"]) #short signal
        ESw.dropna(inplace=True)
        signal_cols_w.append(f"{lname} {sname} signal")

In [ ]:
signal_cols_w

In [ ]:
ESw = ESw.copy()
ESw

# Exponential Moving Average

In [ ]:
def exponential_moving_average(data, column_name, period):
    """
    There could be something weird about this function.
    Frustratingly, it returns a number of values that is one less than
    the other two moving averages. May need to investigate, but keep
    moving for now.
    
    """
    # Calculate the weighting multiplier
    alpha = 2 / (period + 1)

    # Calculate the initial EMA value using a simple moving average
    sma = data[column_name].rolling(window=period, min_periods=period).mean()
    ema_init = sma[period-1]
    
    # Calculate the EMA values for the remaining data
    ema_values = []
    for i in range(period, len(data)):
        ema = alpha * data[column_name][i] + (1 - alpha) * ema_init
        ema_init = ema
        ema_values.append(ema)

    ema_values = [float('nan')] * (period) + ema_values
    data[f"{period}_lema"] = ema_values

In [ ]:
def exponential_moving_average_s(data, column_name, period):
    """
    There could be something weird about this function.
    Frustratingly, it returns a number of values that is one less than
    the other two moving averages. May need to investigate, but keep
    moving for now.
    
    """
    # Calculate the weighting multiplier
    alpha = 2 / (period + 1)

    # Calculate the initial EMA value using a simple moving average
    sma = data[column_name].rolling(window=period, min_periods=period).mean()
    ema_init = sma[period-1]
    
    # Calculate the EMA values for the remaining data
    ema_values = []
    for i in range(period, len(data)):
        ema = alpha * data[column_name][i] + (1 - alpha) * ema_init
        ema_init = ema
        ema_values.append(ema)

    ema_values = [float('nan')] * (period) + ema_values
    data[f"{period}_sema"] = ema_values

In [ ]:
# Just for reference
# LongMAs = list(range(1, 51)) # change this back to 51
# SignalMAs = list(range(1, 51))
# prix_met = 'Open'
# prix_met_2 = 'Close'

for ma in LongMAs :
    exponential_moving_average_s(ESe, 'Open', ma)

for ma in SignalMAs :
    exponential_moving_average(ESe, 'Open', ma) 

In [ ]:
ESe = ESe.copy()
ESe

In [ ]:
l_ema_names = []
for period in LongMAs :
    l_ema_names.append(f"{period}_lema")

s_ema_names = []
for period in SignalMAs :
    s_ema_names.append(f"{period}_sema")

In [ ]:
l_ema_names

In [ ]:
s_ema_names

In [ ]:
# This is the hog
signal_cols_e = []

for lname in l_ema_names :
    for sname in s_ema_names :
#         if pd.to_numeric(re.findall(r'\d+', lname)) > pd.to_numeric(re.findall(r'\d+', sname)) :
        ESe[f"{lname} {sname} signal"] = np.where(ESe[sname] > ESe[lname], 1, 0) #long signal
        ESe[f"{lname} {sname} signal"] = np.where(ESe[sname] < ESe[lname], -1, ESe[f"{lname} {sname} signal"]) #short signal
        ESe.dropna(inplace=True)
        signal_cols_e.append(f"{lname} {sname} signal")

In [ ]:
ESe = ESe.copy()
ESe

# Profit Calculations

### The following code uses my own methods for calculating profit based on point accrual.

In [ ]:
#running this code more than once in the kernal screws up the affected columns; 
#need to move the signal down one row with shift() if we use Open to Open differences
#could also possibly shift point_profit

# for sig_col in signal_cols :
#     ES[sig_col] = ES[sig_col].shift()

In [ ]:
ES['movement'] = ES['Open'].diff().shift(-1) #this line calculates movement from Open to Open
ESw['movement'] = ESw['Open'].diff().shift(-1)
ESe['movement'] = ESe['Open'].diff().shift(-1)
#ES['movement'] = ES['Close'] - ES['Open']

In [ ]:
profit_cols = []
for sig_col in signal_cols :
    ES[f"{sig_col} point profit"] = ES['movement']*ES[sig_col]
    profit_cols.append(f"{sig_col} point profit")
    
ES.dropna(inplace=True) #drops records that we can't really use

In [ ]:
profit_cols_w = []
for sig_col in signal_cols_w :
    ESw[f"{sig_col} point profit"] = ESw['movement']*ESw[sig_col]
    profit_cols_w.append(f"{sig_col} point profit")
    
ESw.dropna(inplace=True) #drops records that we can't really use

In [ ]:
profit_cols_e = []
for sig_col in signal_cols_e :
    ESe[f"{sig_col} point profit"] = ESe['movement']*ESe[sig_col]
    profit_cols_e.append(f"{sig_col} point profit")
    
ESe.dropna(inplace=True) #drops records that we can't really use

In [ ]:
profit_cols
# signal_cols
# ES

In [ ]:
profit_cols_w

In [ ]:
profit_cols_e

In [ ]:
#number of e-mini contracts traded (one micro is one tenth of an e-mini, a full contract is five e-minis)
#full value: $250
#e-mini value: $50
#micro value: $5

contracts = 1
point_value = 50

#buy/hold futures dollars
dollars_bh = ((ES[prix_met][-1])-(ES[prix_met][0]))*point_value*contracts #multiply difference of now and then by $50/pt/contract
dollars_bh = "${:.2f}".format(dollars_bh) #format the result in dollars

#system futures dollars
profit_totals = []
for prof_col in profit_cols :
    profit_totals.append(ES[prof_col].sum()*point_value*contracts)
    
    print('Returns for '+prof_col.replace(' signal point profit', '')+ ' Strategy')
    print("${:.2f}".format(ES[prof_col].sum()*point_value*contracts))
    print()
# print('Buy and Hold Strategy Returns')
# print(dollars_bh)     

In [ ]:
profit_totals_w = []
for prof_col in profit_cols_w :
    profit_totals_w.append(ESw[prof_col].sum()*point_value*contracts)
    
    print('Returns for '+prof_col.replace(' signal point profit', '')+ ' Strategy')
    print("${:.2f}".format(ESw[prof_col].sum()*point_value*contracts))
    print()

In [ ]:
profit_totals_e = []
for prof_col in profit_cols_e :
    profit_totals_e.append(ESe[prof_col].sum()*point_value*contracts)
    
    print('Returns for '+prof_col.replace(' signal point profit', '')+ ' Strategy')
    print("${:.2f}".format(ESe[prof_col].sum()*point_value*contracts))
    print()

In [ ]:
profit_totals

In [ ]:
profit_totals_w

In [ ]:
profit_totals_e

In [ ]:
# Create column name list.
col_names =  ['strategy', 'profit']
  
# Create an empty dataframe with columns.
strategy_performances  = pd.DataFrame(columns = col_names)
strategy_performances_w = strategy_performances.copy()
strategy_performances_e = strategy_performances.copy()

# Show the dataframe.
strategy_performances

In [ ]:
strategy_performances.strategy = signal_cols
strategy_performances.profit = profit_totals

strategy_performances_w.strategy = signal_cols_w
strategy_performances_w.profit = profit_totals_w

strategy_performances_e.strategy = signal_cols_e
strategy_performances_e.profit = profit_totals_e

In [ ]:
strat_perfs = strategy_performances.sort_values(by=['profit'], ascending = False)
strat_perfs_w = strategy_performances_w.sort_values(by=['profit'], ascending = False)
strat_perfs_e = strategy_performances_e.sort_values(by=['profit'], ascending = False)

In [ ]:
strat_perfs

In [ ]:
strat_perfs_w

In [ ]:
strat_perfs_e

In [ ]:
strat_perfs[['lma', 'sma']] = strat_perfs['strategy'].str.split(' ', 1, expand=True)
strat_perfs[['lma', 'junk']] = strat_perfs['lma'].str.split('_', 1, expand=True)
strat_perfs[['sma', 'junk2']] = strat_perfs['sma'].str.split('_', 1, expand=True)

In [ ]:
strat_perfs_w[['lwma', 'swma']] = strat_perfs_w['strategy'].str.split(' ', 1, expand=True)
strat_perfs_w[['lwma', 'junk']] = strat_perfs_w['lwma'].str.split('_', 1, expand=True)
strat_perfs_w[['swma', 'junk2']] = strat_perfs_w['swma'].str.split('_', 1, expand=True)

In [ ]:
strat_perfs_e[['lema', 'sema']] = strat_perfs_e['strategy'].str.split(' ', 1, expand=True)
strat_perfs_e[['lema', 'junk']] = strat_perfs_e['lema'].str.split('_', 1, expand=True)
strat_perfs_e[['sema', 'junk2']] = strat_perfs_e['sema'].str.split('_', 1, expand=True)

In [ ]:
strat_perfs.drop(columns = ['junk','junk2'], inplace = True)
strat_perfs

In [ ]:
strat_perfs_w.drop(columns = ['junk','junk2'], inplace = True)
strat_perfs_w

In [ ]:
strat_perfs_e.drop(columns = ['junk','junk2'], inplace = True)
strat_perfs_e

In [ ]:
strat_perfs.dtypes

In [ ]:
to_num = ['lma','sma']

for col in to_num :
    strat_perfs[col] = pd.to_numeric(strat_perfs[col])

strat_perfs.dtypes

In [ ]:
to_num = ['lwma','swma']

for col in to_num :
    strat_perfs_w[col] = pd.to_numeric(strat_perfs_w[col])

strat_perfs_w.dtypes

In [ ]:
to_num = ['lema','sema']

for col in to_num :
    strat_perfs_e[col] = pd.to_numeric(strat_perfs_e[col])

strat_perfs_e.dtypes

In [ ]:
strat_perfs

In [ ]:
plt.rcParams['figure.figsize'] = 20,15 #sets size of the plot
X = strat_perfs.lma
Y = strat_perfs.sma
Z = strat_perfs.profit
data = pd.DataFrame({'X': X, 'Y': Y, 'Z': Z})
data_pivoted = data.pivot("X", "Y", "Z")
ax = sns.heatmap(data_pivoted)
plt.show()

In [ ]:
plt.rcParams['figure.figsize'] = 20,15 #sets size of the plot
X = strat_perfs_w.lwma
Y = strat_perfs_w.swma
Z = strat_perfs_w.profit
data = pd.DataFrame({'X': X, 'Y': Y, 'Z': Z})
data_pivoted = data.pivot("X", "Y", "Z")
ax = sns.heatmap(data_pivoted)
plt.show()

In [ ]:
plt.rcParams['figure.figsize'] = 20,15 #sets size of the plot
X = strat_perfs_e.lema
Y = strat_perfs_e.sema
Z = strat_perfs_e.profit
data = pd.DataFrame({'X': X, 'Y': Y, 'Z': Z})
data_pivoted = data.pivot("X", "Y", "Z")
ax = sns.heatmap(data_pivoted)
plt.show()

In [ ]:
# THINGS TO WORK ON
# 2.FIND A WAY TO ITERATE THROUGH LIST OF TIME PERIODS
# 3.CALCULATE AND IMPLEMENT WMA'S
# 5.CALCULATE TOTAL TRANSACTION FEES
# 6.INVESTIGATE THAT SYSTEM RESULTS ARE CORRECT
# 7.FIND A WAY TO INCORPORATE TICK/VOLUME CANDLE DATA IN ANY WAY POSSIBLE
# 8.LEARN HOW TO USE A TRADING API
# 9.LEARN TO USE JAVASCRIPT/C# IF NECESSARY
# 11.CONSIDER ADDING THE COLUMNS AND ENRICHING THE DATA IN THE WAYS YOU DREAMT ABOUT
#    -DIFFERENCE BETWEEN SMA & LMA
#     -RELATIVE VOLUME OR DIFFERENCE BETWEEN VOLUME AND VMA
#      -MOVING AVERAGE RIBBON AND SUM OF DIFFERENCE
#       -RSI CALCULATION
#        -LOCAL HIGHS AND LOWS
#         -VWAP CALCULATION
#          -
# 12. CONSIDER CONCATENATING THE MOVING AVERAGE VALUES TO MAKE THE TABLE SMALLER
#    -YOU WILL HAVE TO FIGURE OUT HOW TO ACCESS THESE SUBSTRINGS FOR CALCULATING SIGNALS

### VVV Experimental Code